In [113]:
import pandas as pd
import csv
import matplotlib.pyplot as plt
import numpy as np
from IPython import display

# get df from csv (in local directory)
df = pd.read_csv(r'C:\Users\jeong\research\grading-data-6-17-22.csv')


# general tools

# returns filtered dataframe. Each condition should be passed as column name = LIST of targets
    # e.g. "filter(df, crsTitle = ['PHYSICS I LAB'], facultyID = ['F18125', 'F97128'])" returns df with 84 rows
def filter(df, **kwargs):
    for key in kwargs.keys():
        df = df[(df[key]).isin(kwargs.get(key))]
    return df

# check if object is nan
def isNaN(string):
    return string != string

# returns a dictionary of "facultyID: number of unique CRNs attributed to this instructor"
def total_courses_per_instructor(df):
    faculty_list = np.unique(list(df['facultyID']))
    faculty_courses = { i : 0 for i in faculty_list }
    for i in faculty_list:
        faculty_history = filter(df, facultyID = [i])
        faculty_courses.update({i: (len(np.unique(list((faculty_history)['CRN']))))})
    return faculty_courses


# data cleaning
    # original df remains unchanged. cleaned data is put in new dataframe cleaned_df
cleaned_df = df.copy()
cleaned_df.replace(" ", np.nan, inplace=True)
cleaned_df['finGradN'] = cleaned_df['finGradN'].astype('float')
cleaned_df['facultyID']=cleaned_df['facultyID'].astype(str)
# drop administrative CBA/FCRH as they are not actual students
cleaned_df.drop(cleaned_df[cleaned_df['ProgCode'] == 'Administrative CBA'].index,  inplace = True)
cleaned_df.drop(cleaned_df[cleaned_df['ProgCode'] == 'Administrative FCRH'].index, inplace = True)
# drop rows without course title
cleaned_df.drop(cleaned_df[isNaN(cleaned_df['crsTitle'])].index, inplace = True)
# creates a new CRN: CRN + last two digits of year + one digit based on semester
    # e.g. oldCRN 11135, Summer 2010 course -> CRN: 11135102
cleaned_df[['sem', 'year']] = cleaned_df['term'].str.split(' ', n = 1, expand = True)
cleaned_df['year'] = pd.to_numeric(cleaned_df['year'])
cleaned_df['year'] = cleaned_df.apply(lambda x: x['year'] % 1000, axis=1)
cleaned_df.loc[cleaned_df['sem'] == 'Fall', 'sem'] = 0
cleaned_df.loc[cleaned_df['sem'] == 'Spring', 'sem'] = 1
cleaned_df.loc[cleaned_df['sem'] == 'Summer', 'sem'] = 2
cleaned_df['sem'] = pd.to_numeric(cleaned_df['sem'])
cleaned_df.rename(columns = {'CRN':'oldCRN'}, inplace = True)
cleaned_df['oldCRN'] = cleaned_df['oldCRN'].astype(str)
cleaned_df['year'] = cleaned_df['year'].astype(str)
cleaned_df['sem'] = cleaned_df['sem'].astype(str)
cleaned_df['CRN'] = cleaned_df['oldCRN'] + cleaned_df['year']+ cleaned_df['sem']


import difflib
courses_raw = np.unique(list(cleaned_df['crsTitle']))
for_manual_comparison = []
for i in list(range(len(courses_raw))):
    for j in list(range(i+1, len(courses_raw))):
        seq = difflib.SequenceMatcher(None, courses_raw[i], courses_raw[j])
        if 0.8 < seq.ratio() < 1.0:
            for_manual_comparison.append([courses_raw[i], courses_raw[j]])

def mode(arr):
        if arr==[]:
            return None
        else:
            return max(set(arr), key=arr.count)

to_delete = []
for i in list(range((len(for_manual_comparison)))):
    lst0 = mode(list((filter(cleaned_df, crsTitle = [(for_manual_comparison[i])[0]]))['NumCode']))
    lst1 = mode(list((filter(cleaned_df, crsTitle = [(for_manual_comparison[i])[1]]))['NumCode']))
    if lst0 != lst1:
        to_delete.append(for_manual_comparison[i])

lang_courses = ['ARABIC', 'ITALIAN', 'JAPANESE', 'LATIN', 'MANDARIN', 'RUSSIAN', 'FRENCH', 'GREEK', 'GERMAN', 'SPANISH']
for i in lang_courses:
    for j in list(range(len(for_manual_comparison))):
        if i in (for_manual_comparison[j])[0]:
            if (not(i in (for_manual_comparison[j])[1])):
                to_delete.append(for_manual_comparison[j])

for_manual_comparison = [x for x in for_manual_comparison if x not in to_delete]

# all EP/non EP courses were combined
final_touches = [['ARCHITECTURAL DESIGN', 'URBAN ARCHITECTURAL DESIGN I'], 
['CHRISTIAN THOUGHT& PRACTICE I', 'CHRISTIAN THOUGHT& PRACTICE II'], 
['CONTEMPORARY ETHICS', 'CONTEMPORARY MUSIC'],
['DIRECTING PRODUCTION WKSHOP I', 'DIRECTING PRODUCTION WKSHOP II'], 
['DIRECTING PRODUCTION WKSHOPII', 'DIRECTING PRODUCTION WKSHOPIII'],
['INTRO TO POLITICAL PHIL', 'INTRO TO POLITICAL THEORY'],
['INTRO TO POLITICAL PHILOSOPHY', 'INTRO TO POLITICAL THEORY'],
['JOURNALISM WORKSHOP', 'JOURNALISM WORKSHOP: LAYOUT'],
['JOURNALISM WORKSHOP', 'JOURNALISM WORKSHOP: PHOTO'],
['JOURNALISM WORKSHOP: LAYOUT', 'JOURNALISM WORKSHOP: PHOTO']]

for i in final_touches:
    if i in for_manual_comparison:
        for_manual_comparison.remove(i)

final_df = cleaned_df.copy()
remove_dup_dict = {}
for i in for_manual_comparison:
    if len(i[0]) > len(i[1]):
        longer = i[0]
        shorter = i[1]
    else:
        longer = i[1]
        shorter = i[0]
    remove_dup_dict.update({longer: shorter})

remove_dup_dict.update({'"RACE" AND "MIXED RACE"': 'ACE" AND "MIXED RACE""'})
remove_dup_dict.update({'AFRICAN AMERICAN HISTORY I': 'AFRICAN-AMERICAN HISTORY I'})
remove_dup_dict.update({'FRENCH THEATER AND PERFORMANCE': 'FRENCH THEATRE/ PERFORMANCE'})
remove_dup_dict.update({'FRENCH THEATER AND PERFORMANCE': 'FRENCH THEATRE/PERFORMANCE'})
remove_dup_dict.update({'PHILOSOPHICAL  ETHICS': 'PHILOSOPHICAL ETHICS (EP3)'})

for i in remove_dup_dict.values():
    final_df.loc[final_df['crsTitle'] == i, 'crsTitle'] = list(remove_dup_dict.keys())[list(remove_dup_dict.values()).index(i)]

# data is all cleaned and saved in dataframe final_df

In [123]:
# returns Average, SD, Z-score of GPA of one section, for a particular subject course_name, for sections with more than class_size students
    # e.g. gpa_by_section(final_df, 'TEXTS AND CONTEXTS', 6) returns a dataframe of columns: CRN, Instructor, GPA, SD, Class Size, Z-score
    # class size was separately computed. (did not use class_size column in original df as it was inaccurate.)
def gpa_by_section(df, course_name, class_size):
    sections = list(np.unique((filter(df, crsTitle = [course_name])['CRN'])))
    to_return = pd.DataFrame(columns =['CRN', 'Instructor', 'GPA', 'SD', 'Class Size'])
    
    for i in sections:
        filtered_data = filter(df, crsTitle = [course_name], CRN = [i])
        to_return.loc[len(to_return)] = [i, (filtered_data.mode()['facultyID'][0]), np.mean(((filtered_data)['finGradN']).to_numpy()), np.std(((filtered_data)['finGradN']).to_numpy()), len(filtered_data.index)]

    to_return = to_return[to_return['Class Size'] >= class_size]

    ttl_mean = (np.mean(to_return.GPA.values.tolist()))
    ttl_sd = (np.std(to_return.GPA.values.tolist()))
    to_return['Z-score'] = (to_return['GPA']-ttl_mean)/ttl_sd

    return to_return

# creates list of course names in order of highest occurences in dataframe
course_data_points = {}
for i in list(final_df['crsTitle']):
  course_data_points[i] = course_data_points.get(i, 0) + 1
sorted_tuples = sorted(course_data_points.items(), key=lambda item: item[1], reverse=True)
courses_in_order = list(({k: v for k, v in sorted_tuples}).keys())

# list of unique instructors (facultyID) / courses (crsTitle)
instructors = [k for (k,v) in total_courses_per_instructor(final_df).items() if 5 <= v <= 100]
#instructors = np.unique(list(final_df['facultyID']))
courses = np.unique(list(final_df['crsTitle']))

# dictionary of instructors that taught sections with exceptionally high/low grades
easy_instructor_tally = { i : 0 for i in instructors }
hard_instructor_tally = { i : 0 for i in instructors }

# loops through every course and adds +1 to the value of the instructor in appropriate dictionary if a section they taught had exceptionally high/low grades
# then, prints out instructors who were flagged for more than a certain percentage of the courses they taught
    # modify below parameters as needed. 
target_course_list = courses_in_order[:200]     # which master list of courses to go through
z_score_cutoff = 1.5    # Z-score positively and negatively to flag section
flag_percentage = 70    # Percentage of sections that the instructor taught that need to be flagged for final list result

# run time tracker
current_index = 0
total_index = len(target_course_list)

for i in target_course_list:
    # disregards sections with class size less than two standard deviations below the average class size of that course
    all_class_sizes = (list(gpa_by_section(final_df, i, 0)['Class Size']))
    all_sections = gpa_by_section(final_df, i, (np.mean(all_class_sizes))-2*(np.std(all_class_sizes)))

    # "exceptionally low or high" defined as z_score_cutoff standard deviations
    easy_anomalies = all_sections[all_sections['Z-score'] > z_score_cutoff]
    if len(easy_anomalies.index) > 0 : 
        for j in list(easy_anomalies['Instructor']):
            if not (isNaN(j)):
                if j in instructors:
                    easy_instructor_tally[j]+=1

    hard_anomalies = all_sections[all_sections['Z-score'] < -z_score_cutoff]
    if len(hard_anomalies.index) > 0 : 
        for k in list(hard_anomalies['Instructor']):
            if not (isNaN(k)):
                if k in instructors:
                    hard_instructor_tally[k]+=1

    # run time tracker
    current_index+=1
    print("current index: "+str(current_index)+" out of "+str(total_index))

num_courses_taught_per_instructor = total_courses_per_instructor(final_df)

print([k for k, v in easy_instructor_tally.items() if v > (num_courses_taught_per_instructor[k])*(flag_percentage/100)])
print([k for k, v in hard_instructor_tally.items() if v > (num_courses_taught_per_instructor[k])*(flag_percentage/100)])

current index: 1 out of 200
current index: 2 out of 200
current index: 3 out of 200
current index: 4 out of 200
current index: 5 out of 200
current index: 6 out of 200
current index: 7 out of 200
current index: 8 out of 200
current index: 9 out of 200
current index: 10 out of 200
current index: 11 out of 200
current index: 12 out of 200
current index: 13 out of 200
current index: 14 out of 200
current index: 15 out of 200
current index: 16 out of 200
current index: 17 out of 200
current index: 18 out of 200
current index: 19 out of 200
current index: 20 out of 200
current index: 21 out of 200
current index: 22 out of 200
current index: 23 out of 200
current index: 24 out of 200
current index: 25 out of 200
current index: 26 out of 200
current index: 27 out of 200
current index: 28 out of 200
current index: 29 out of 200
current index: 30 out of 200
current index: 31 out of 200
current index: 32 out of 200
current index: 33 out of 200
current index: 34 out of 200
current index: 35 out o

In [115]:
# returns Average, SD, Z-score of GPA of one section, for a particular subject course_name, for sections with more than class_size students
    # e.g. gpa_by_section(final_df, 'TEXTS AND CONTEXTS', 6) returns a dataframe of columns: CRN, Instructor, GPA, SD, Class Size, Z-score
    # class size was separately computed. (did not use class_size column in original df as it was inaccurate.)
def gpa_by_section(df, course_name, class_size):
    sections = list(np.unique((filter(df, crsTitle = [course_name])['CRN'])))
    to_return = pd.DataFrame(columns =['CRN', 'Instructor', 'GPA', 'SD', 'Class Size'])
    
    for i in sections:
        filtered_data = filter(df, crsTitle = [course_name], CRN = [i])
        to_return.loc[len(to_return)] = [i, (filtered_data.mode()['facultyID'][0]), np.mean(((filtered_data)['finGradN']).to_numpy()), np.std(((filtered_data)['finGradN']).to_numpy()), len(filtered_data.index)]

    to_return = to_return[to_return['Class Size'] >= class_size]

    ttl_mean = (np.mean(to_return.GPA.values.tolist()))
    ttl_sd = (np.std(to_return.GPA.values.tolist()))
    to_return['Z-score'] = (to_return['GPA']-ttl_mean)/ttl_sd

    return to_return


# list of unique instructors (facultyID) / courses (crsTitle)
instructors = np.unique(list(final_df['facultyID']))
courses = np.unique(list(final_df['crsTitle']))

# dictionary of instructors that taught sections with exceptionally high/low grades
easy_instructor_tally = { i : 0 for i in instructors }
hard_instructor_tally = { i : 0 for i in instructors }

# run time tracker
current = 0
total_index = len(courses)

# loops through every course and adds +1 to the value of the instructor in appropriate dictionary if a section they taught had exceptionally high/low grades
# then, prints out instructors who were flagged for more than a certain percentage of the courses they taught
    # modify below parameters as needed. 
z_score_cutoff = 2.0    # Z-score positively and negatively to flag section
flag_percentage = 70    # Percentage of sections that the instructor taught that need to be flagged for final list result
target_df = courses     # which master list of courses to go through

for i in target_df:
    # disregards sections with class size less than two standard deviations below the average class size of that course
    all_class_sizes = (list(gpa_by_section(final_df, i, 0)['Class Size']))
    all_sections = gpa_by_section(final_df, i, (np.mean(all_class_sizes))-2*(np.std(all_class_sizes)))

    # "exceptionally low or high" defined as z_score_cutoff standard deviations
    easy_anomalies = all_sections[all_sections['Z-score'] > z_score_cutoff]
    if len(easy_anomalies.index) > 0 : 
        for j in list(easy_anomalies['Instructor']):
            if not (isNaN(j)):
                easy_instructor_tally[j]+=1

    hard_anomalies = all_sections[all_sections['Z-score'] < -z_score_cutoff]
    if len(hard_anomalies.index) > 0 : 
        for k in list(hard_anomalies['Instructor']):
            if not (isNaN(k)):
                hard_instructor_tally[k]+=1

    # run time tracker
    print("Current index: "+str(current)+" out of "+str(total_index))
    current+=1

num_courses_taught_per_instructor = total_courses_per_instructor(final_df)

print([k for k, v in easy_instructor_tally.items() if v > (num_courses_taught_per_instructor[k])*(flag_percentage/100)])
print([k for k, v in hard_instructor_tally.items() if v > (num_courses_taught_per_instructor[k])*(flag_percentage/100)])

Current index: 0 out of 2516
Current index: 1 out of 2516
Current index: 2 out of 2516
Current index: 3 out of 2516
Current index: 4 out of 2516
Current index: 5 out of 2516
Current index: 6 out of 2516
Current index: 7 out of 2516
Current index: 8 out of 2516
Current index: 9 out of 2516
Current index: 10 out of 2516
Current index: 11 out of 2516
Current index: 12 out of 2516
Current index: 13 out of 2516
Current index: 14 out of 2516
Current index: 15 out of 2516
Current index: 16 out of 2516
Current index: 17 out of 2516
Current index: 18 out of 2516
Current index: 19 out of 2516
Current index: 20 out of 2516
Current index: 21 out of 2516
Current index: 22 out of 2516
Current index: 23 out of 2516
Current index: 24 out of 2516
Current index: 25 out of 2516
Current index: 26 out of 2516
Current index: 27 out of 2516
Current index: 28 out of 2516
Current index: 29 out of 2516
Current index: 30 out of 2516
Current index: 31 out of 2516
Current index: 32 out of 2516
Current index: 33 ou

KeyboardInterrupt: 

In [ ]:
# visualizations

# histogram of number of courses taught by instructors (0 to 100 courses)
    # note: the only instructor who taught >100 sections is "F46611" who taught 216 sections.
    # we can see that nearly half the instructors taught <5 courses. 
filtered = {k:v for (k,v) in num_courses_taught_per_instructor.items() if v <= 100}
plt.figure(figsize=(15, 10))
values, bins, bars = plt.hist(filtered.values(), bins= np.arange(0, 100, 5), edgecolor='black', linewidth=0.5)
plt.xlabel("num courses")
plt.ylabel("num instructors")
plt.xticks((range(0, 100, 5)))
plt.bar_label(bars, fontsize=10, color='navy')
plt.show()
